# Google books апи

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import logging

logging.config.fileConfig('logging.ini', defaults={'logfilename': 'googleapi.log'})


In [5]:
logger = logging.getLogger("sLogger")

На этом этапе мы хотим обогатить датасеты информаций о книгах, полученных с помощью google books api

In [6]:
import os
GOOGLE_API_KEYS = os.environ.get("GOOGLE_API_KEYS").split(',')

logger.info(f'Найдено {len(GOOGLE_API_KEYS)} ключей')
GOOGLE_URL = "https://www.googleapis.com/books/v1/volumes"

In [7]:
import requests
import time
import pandas as pd

df = pd.read_csv("final.csv")
df.head()

,Unnamed: 0,title,author,isbn,pages,publication_year,rating,reviews_count,price,currency,age_restriction,genres,publisher,formats,url,price_numeric,rating_numeric,pages_numeric,reviews_numeric
0,0,По осколкам твоего сердца,Анна Джейн,NaN,500.0,2025.0,4.9,3530.0,229.0,RUB,16+,"Литрес Авторы, От ненависти до любви, Первая л...",Правообладатель: Автор,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/anna-dzheyn/po-osko...,229.0,4.9,500.0,3530.0
1,1,Твое сердце будет разбито,Анна Джейн,NaN,510.0,2022.0,4.8,7228.0,199.0,RUB,16+,"Литрес Авторы, Любовь и ненависть, Молодежные ...",Правообладатель: Автор,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/anna-dzheyn/tvoe-se...,199.0,4.8,510.0,7228.0
2,2,Фейерверк на ладони,Ольга Назарова,NaN,430.0,2026.0,4.9,139.0,189.0,RUB,16+,"Антистресс, Ироничная проза, Книги для души, К...",Правообладатель: Автор,"epub,, fb2,, fb3,, ios.epub,, mobi,, pdf,, txt...",https://www.litres.ru/book/olga-stanislavovna-...,189.0,4.9,430.0,139.0
3,3,Проект «Аве Мария»,Энди Вейер,NaN,21.0,2021.0,4.8,2117.0,529.0,RUB,16+,"Авантюрные приключения, Близкое будущее, Заруб...",Правообладатель: Аудио-ЛАУ,"m4b,, mp3,, zip",https://www.litres.ru/audiobook/endi-veyer/pro...,529.0,4.8,21.0,2117.0
4,4,По осколкам твоего сердца,Анна Джейн,NaN,14.0,2022.0,4.8,921.0,749.0,RUB,16+,"Young adult, Книги о подростках, Любовь и нена...",Правообладатель: Издательство CLEVER,"m4b,, mp3,, zip",https://www.litres.ru/audiobook/anna-dzheyn/po...,749.0,4.8,14.0,921.0


In [8]:
df_nyt = pd.read_csv('final_nyt_clean.csv')
df_nyt.head()

,Unnamed: 0.1,Unnamed: 0,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,...,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links,avg_rating,rating_count,title_open_library,author_open_library
0,9791,38,NaN,https://www.amazon.com/dp/125031397X?tag=thene...,NaN,0,Aiden Thomas,https://static01.nyt.com/bestsellers/images/97...,500,323,...,NaN,LOST IN THE NEVER WOODS,2026-03-06T06:19:00.358Z,1,"[{'isbn10': '', 'isbn13': '9781250313973'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",1.0,1.0,Lost in the Never Woods,"['Aiden Thomas', 'Avi Roque']"
1,9453,491,NaN,https://www.amazon.com/dp/1464223335?tag=thene...,NaN,0,Ana Huang,https://static01.nyt.com/bestsellers/images/97...,500,324,...,NaN,THE DEFENDER,2026-03-05T17:47:52.965Z,1,"[{'isbn10': '', 'isbn13': '9781464223334'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",1.0,1.0,The Defender,['Ana Huang']
2,62,62,NaN,https://www.amazon.com/dp/059344129X?tag=thene...,NaN,0,Emily Henry,https://static01.nyt.com/bestsellers/images/97...,500,331,...,NaN,GREAT BIG BEAUTIFUL LIFE,2026-03-18T00:52:24.385Z,0,"[{'isbn10': '', 'isbn13': '9798217063987'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",1.0,1.0,Great Big Beautiful Life,['Emily Henry']
3,1981,1366,NaN,https://www.amazon.com/dp/0062937405?tag=thene...,NaN,0,Meena Harris,https://static01.nyt.com/bestsellers/images/97...,500,389,...,NaN,KAMALA AND MAYA'S BIG IDEA,2026-03-06T06:03:32.289Z,1,"[{'isbn10': '', 'isbn13': '9780062937407'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",1.0,1.0,Kamala and Maya’s Big Idea,"['Meena Harris', 'Ana Ramírez González']"
4,2290,1675,NaN,https://www.amazon.com/dp/0063221942?tag=thene...,NaN,0,Jenna Kutcher,https://static01.nyt.com/bestsellers/images/97...,500,331,...,NaN,"HOW ARE YOU, REALLY?",2026-03-06T05:18:30.124Z,0,"[{'isbn10': '', 'isbn13': '9780063221949'}]","[{'name': 'Amazon', 'url': 'https://www.amazon...",1.0,1.0,"How Are You, Really?",['Jenna Kutcher']


Поиск информации о книжке по ее названию и автору

In [9]:
def search_book(title, author, api_key):
    responce = requests.get(GOOGLE_URL, params={"q": f"intitle:{title}+inauthor:{author}", "maxResults": 5, "key": api_key})
    return responce.status_code, responce.json()

Тестовый запрос

In [10]:
search_book("Маленькая хозяйка большой кухни-3", "Наталья Лакота", GOOGLE_API_KEYS[-1])

(200,
 {'kind': 'books#volumes',
  'totalItems': 2,
  'items': [{'kind': 'books#volume',
    'id': 'DI-4EQAAQBAJ',
    'etag': '57jMEUkodMI',
    'selfLink': 'https://www.googleapis.com/books/v1/volumes/DI-4EQAAQBAJ',
    'volumeInfo': {'title': 'Маленькая хозяйка большой кухни-3',
     'authors': ['Наталья Лакота'],
     'publisher': 'ЛитРес, SelfPub',
     'publishedDate': '2026-01-26',
     'description': 'Третья и заключительная история про Сесилию Лайон – маленькую хозяйку королевской кухни. Сесилия назначена инспектрисой королевской кухни, Ричард де Морвиль снова занял должность маршала при дворе, и всё складывается как нельзя лучше. Но вдруг снова тревожно звучит колокол. На этот раз... свадебный.',
     'industryIdentifiers': [{'type': 'ISBN_13',
       'identifier': '9785048628386'},
      {'type': 'ISBN_10', 'identifier': '5048628381'}],
     'readingModes': {'text': True, 'image': True},
     'pageCount': 346,
     'printType': 'BOOK',
     'categories': ['Fiction'],
     'm

Получим только необходимые для дальнейшего анализа поля

In [54]:
def search_correct_book(title, author, api_key):
    status_code, all_books = search_book(title, author, api_key)
    if not all_books.get('items'):
        logger.info(f'Книга {title} пока не найдена. Код ответа: {status_code}')
        return status_code, [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
    
    first_book = all_books['items'][0]
    id_book = first_book['id']
    categories = first_book.get('volumeInfo', dict()).get('categories', None)
    publisher = first_book.get('volumeInfo', dict()).get('publisher', None)
    description = first_book.get('volumeInfo', dict()).get('description', None)
    print_type = first_book.get('volumeInfo', dict()).get('print_type', None)
    page_count = first_book.get('volumeInfo', dict()).get('pageCount', None)
    language = first_book.get('volumeInfo', dict()).get('language', None)
    country = first_book.get('saleInfo', dict()).get('country', None)
    saleability = first_book.get('saleInfo', dict()).get('saleability', None)
    is_ebook = first_book.get('saleInfo', dict()).get('isEbook', None)
    retail_price = first_book.get('saleInfo', dict()).get('retailPrice', dict()).get('amount', None)
    currency_code = first_book.get('saleInfo', dict()).get('listPrice', dict()).get('currencyCode', None)
    public_domain = first_book.get('accessInfo', dict()).get('publicDomain', None)
    epub_avaliable = first_book.get('accessInfo', dict()).get('epub', dict()).get('isAvailable', None)
    pdf_avaliable = first_book.get('accessInfo', dict()).get('pdf', dict()).get('isAvailable', None)
    viewability = first_book.get('accessInfo', dict()).get('viewability', None)
    logger.info(f'Книга {title} найдена')
    return status_code, [id_book, categories, publisher, description, print_type, page_count, language, country, saleability, is_ebook, retail_price, currency_code, viewability, public_domain, epub_avaliable, pdf_avaliable]


Тестовый запрос для получения конкретной информации о книге по ее названию и автору

In [55]:
search_correct_book("Маленькая хозяйка большой кухни-3", "Наталья Лакота", GOOGLE_API_KEYS[-1])

(200,
 ['DI-4EQAAQBAJ',
  ['Fiction'],
  'ЛитРес, SelfPub',
  'Третья и заключительная история про Сесилию Лайон – маленькую хозяйку королевской кухни. Сесилия назначена инспектрисой королевской кухни, Ричард де Морвиль снова занял должность маршала при дворе, и всё складывается как нельзя лучше. Но вдруг снова тревожно звучит колокол. На этот раз... свадебный.',
  None,
  346,
  'ru',
  'NL',
  'FOR_SALE',
  True,
  2.7,
  'EUR',
  'PARTIAL',
  False,
  True,
  True])

Дополним исходный датасет новыми столбцами

In [61]:
def merge_df_with_google(df, title, save_throw=500):
    results = []
    new_cols = ['id_book', 'categories', 'publisher', 'description', 'print_type', 'page_count', 'language', 'country', 'saleability', 'is_ebook', 'retail_price', 'currency_code', 'viewability', 'public_domain', 'epub_avaliable', 'pdf_avaliable']
    key = 0
    c = 0
    for i, row in df.iterrows():
        if c % save_throw == 0:
            temp_merged = pd.concat([pd.DataFrame(results, columns=new_cols, index=df.head(c).index), df.head(c)], axis=1)
            temp_merged.to_csv(f'google_x_{title}_to_{c}.csv')
            logger.info(f'Сохранены с 0 по {c} строк')

        try:
            status_code, cols = search_correct_book(row['title'], row['author'], GOOGLE_API_KEYS[key%len(GOOGLE_API_KEYS)])
            while status_code == 429:
                time.sleep(5)
                key += 1
                logger.info(f"Возможно, по этому ключу превышено количество запросов. Пробуем сменить ключ. Используем {key+1} из {len(GOOGLE_API_KEYS)}")
                status_code, cols = search_correct_book(row['title'], row['author'], GOOGLE_API_KEYS[key%len(GOOGLE_API_KEYS)])
            
            results.append(cols)
        except Exception as e:
            logger.exception(e)
            results.append([None]*len(new_cols))
        c += 1

    return pd.concat([pd.DataFrame(results, columns=new_cols, index=df.index), df], axis=1)
    

In [ ]:
logger.info('Начали мерджить книги с Литреса с Гуглом')
merged_final = merge_df_with_google(df, 'litres')
logger.info('Закончили мерджить книги с Литреса с Гуглом')

In [65]:
df.shape

(10070, 19)

In [67]:
merged_final.to_csv('google_x_litres_final_last.csv')

In [68]:
logger.info('Начали мерджить книги с NYT с Гуглом')
merged_nyt = merge_df_with_google(df_nyt, 'nyt')
logger.info('Закончили мерджить книги с NYT с Гуглом')

In [70]:
merged_nyt.to_csv('google_x_nyt_final.csv')